# LoRA / QLoRA / PEFT — Hands-On

## 0. Setup

In [ ]:
%pip install -q numpy
import numpy as np
rng=np.random.RandomState(40); d_in,d_out,r,alpha=6,4,2,8
W=rng.normal(size=(d_out,d_in)); A=rng.normal(scale=.02,size=(r,d_in)); B=rng.normal(scale=.02,size=(d_out,r)); x=rng.normal(size=d_in)

## 1. Forward

In [ ]:
base=W@x; delta=(alpha/r)*(B@(A@x)); y=base+delta
print(np.round(base,4)); print(np.round(delta,6)); print(np.round(y,4))

## 2. Parameter savings

In [ ]:
for rr in [1,2,4,8,16]:
    dense=4096*4096; adapter=rr*(4096+4096)
    print(rr, adapter, round(100*adapter/dense,3))

## 3. Alpha scale

In [ ]:
for a in [1,4,8,16,32]: print(a, round(float(np.linalg.norm((a/r)*(B@(A@x)))),6))

## 4. Merge

In [ ]:
Delta=(alpha/r)*(B@A); print(np.max(np.abs((W+Delta)@x-y))); assert np.allclose((W+Delta)@x,y)

## 5. Quantization intuition

In [ ]:
scale=np.max(np.abs(W))/7; q=np.clip(np.round(W/scale),-8,7).astype(np.int8); deq=q.astype(float)*scale
print(round(float(np.linalg.norm(W-deq)/np.linalg.norm(W)),4), 'float32 bytes', W.size*4, 'int4 bytes', W.size/2)

## 6. Multiple adapters

In [ ]:
for name,(Ai,Bi) in {'legal':(A,B),'support':(A*.5,B*1.5)}.items(): print(name, np.round(W@x+(alpha/r)*(Bi@(Ai@x)),4))

## Exercises
Change rank, simulate wrong base, and compare merge vs hot-swap.